# Watershed DT
Detects if an AR mask overlaps given HUC4 level watersheds in the US, provided there is already a landfalling AR detected.

In [1]:
import xarray as xa
import numpy as np
import regionmask
from pynhd import pynhd
import cftime
import pandas as pd
import geopandas as gpd
from glob import glob
import matplotlib.pyplot as plt

ERROR 1: PROJ: proj_create_from_database: Open of /glade/u/home/tcorrie/.conda/envs/rmask/share/proj failed


In [2]:
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import time
import pandas as pd
import gc
import os

In [3]:
# This might be necessary if you run into HDF5 issues.
#os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [ ]:
# PBSCluster might be overkill for this. A LocalCluster should work if you allocate enough memory upon startup.
# See previous notebooks for examples of setting up LocalCluster.

cluster = PBSCluster(
    account='WYOM0169',
    queue='main',
    cores=4,
    memory='16GB',
    walltime='12:00:00',
    job_extra_directives=['-o /glade/u/home/tcorrie/dask_worker_log/', '-e /glade/u/home/tcorrie/dask_worker_log/'],
    )
cluster.scale(32)
client = Client(cluster)

In [ ]:
print(client)

In [4]:
huc4s = gpd.read_file('/glade/u/home/tcorrie/watersheds_WUS.geojson', driver='GeoJSON')

In [ ]:
def process_timestep(tstep, ardt, model, member, ARmask_future, landfall_future, wshed_mask_future):
    ARmask_ts = ARmask_future.isel(time=tstep)
    landfall_ts = landfall_future.isel(time=tstep)
    ARmask_masked = ARmask_ts.where(ARmask_ts.ARmask == 1)
    wshed_AR_overlap = wshed_mask_future*ARmask_masked.ARmask
    if landfall_ts['landfall_wus'].values == 0:
        return (0, [0]*len(huc4s), wshed_AR_overlap) # Order: 1503, 1605, 1606, 1701-1703, 1705-1712, 1801-1810
        

    def process_watershed(idx):
        total_count = wshed_mask.where(wshed_mask == idx).count().values
        masked_count = wshed_AR_overlap.where(wshed_mask == idx).count().values
        fraction = masked_count/total_count
        #return fraction
        return 1 if fraction >= 0.05 else 0
    
    ARs_by_watershed = [process_watershed(i) for i in range(len(huc4s))]
    return (1, ARs_by_watershed, wshed_AR_overlap)

In [6]:
wusd3 = pd.read_csv('../wusd3_gcms.csv', index_col=0)
for i in range(1011, 1201, 20):
    wusd3.loc[len(wusd3)] = ['cesm2-le', 'CESM2-LE', str(i), '365_day']
wusd3

,Model,Model Name,Member,Calendar
0,access-cm2,ACCESS-CM2,r5i1p1f1,standard
1,canesm5,CanESM5,r1i1p2f1,365_day
2,cesm2,CESM2,r11i1p1f1,365_day
3,cnrm-esm2-1,CNRM-ESM2-1,r1i1p1f2,standard
4,ec-earth3,EC-Earth3,r1i1p1f1,standard
5,ec-earth3-veg,EC-Earth3-Veg,r1i1p1f1,standard
6,era5,ERA5,NaN,standard
7,fgoals-g3,FGOALS-g3,r1i1p1f1,365_day
8,giss-e2-1-g,GISS-E2-1-G,r1i1p1f2,365_day
9,miroc6,MIROC6,r1i1p1f1,standard


In [7]:
# This cell is for parallelization. If your data is small enough or can't get this to run, skip this cell and run everything
#   in serial below.

for row in range(len(wusd3)):
    if row != 6:
        continue

    model = wusd3.loc[row, 'Model']
    member = wusd3.loc[row, 'Member']
    print(f"Watersheds for {model}/{member};")
    
    for ardt in ['G18', 'TE', 'tARget']: #G18, TE, tARget
        if ardt != 'G18':
            continue
        print(f'{ardt}:', end=" ")
        t1 = time.time()
        ARmask = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARmasks/{ardt}/ARmask.{model}.{member}.nc', use_cftime=True, chunks={'time': 1})
        landfall = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARlandfall/{ardt}/ARlandfall.{ardt}.{model}.{member}.nc', chunks={'time': 1})
        wshed_mask = regionmask.mask_geopandas(huc4s, ARmask.lon, ARmask.lat)
        
        times = list(range(len(ARmask.time)))

        ARmask_future = client.scatter(ARmask, broadcast=True)
        landfall_future = client.scatter(landfall, broadcast=True)
        wshed_mask_future = client.scatter(wshed_mask, broadcast=True)

        futures = client.map(process_timestep, times, 
                             ardt=ardt, model=model, member=member, 
                             ARmask_future=ARmask_future,
                             landfall_future=landfall_future,
                             wshed_mask_future = wshed_mask_future
                            )    
        results = client.gather(futures)

        landfall_wus, landfall_watershed, ARmaskwsheds = zip(*results)

        watershed_ds = xa.Dataset(
                        coords=dict(time=('time',times)),
                        data_vars=dict(
                            ARwshed_any=(["time"], np.array(landfall_wus)),
                        ),
                        attrs=dict(description='A binary mask for discerning landfalling ARs over the Western US and split by watershed')
                        
                    )
        for idx, lw in enumerate(landfall_watershed):
            wshedvar = f"ARwshed_{huc4s.loc[idx, 'huc4']}"
            watershed_ds[wshedvar] = (['time'], np.array(lw))

        watershed_mask_ds = xa.Dataset(
            coords=dict(
                time=('time', times),
                lat=('lat', ARmask.lat),
                lon=('lon', ARmask.lon),
            ),
            data_vars=dict(
                wshed_mask=(['time','lat','lon'], np.array(ARmaskwsheds))
            ),
            attrs=dict(description='The points where an AR mask overlaps the any watershed.')
        )

        watershed_ds.to_netcdf(f'/glade/work/tcorrie/ARdata/ARwatershed/{ardt}/ARwshed.{ardt}.{model}.{member}.nc')
        watershed_mask_ds.to_netcdf(f'/glade/work/tcorrie/ARdata/ARwatershedmask/{ardt}/ARwshedmsk.{ardt}.{model}.{member}.nc')
        t2 = time.time()
        print(f'{(t2-t1)//60:.0f}\'{(t2-t1)%60:04.1f}\" run time')
        gc.collect()

Watersheds for era5/nan;
G18: 

NameError: name 'client' is not defined

In [9]:
# Run this in serial if your dataset is small enough.
%%time

for row in range(len(wusd3)):
    if row==6:
        continue
    model = wusd3.loc[row, 'Model']
    member = wusd3.loc[row, 'Member']
    print(f"{model}/{member}:")
    for ardt in ['G18', 'TE', 'tARget']:
        print(ardt)
        ARmask_full = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARmasks/{ardt}/ARmask.{model}.{member}.nc', use_cftime=True).load()
        landfall_full = xa.open_dataset(f'/glade/work/tcorrie/ARdata/ARlandfall/{ardt}/ARlandfall.{ardt}.{model}.{member}.nc')[['landfall_wus']].load()
        wshed_mask = regionmask.mask_geopandas(huc4s, ARmask_full.lon, ARmask_full.lat)
        
        ARmasked = ARmask_full.where(ARmask_full.ARmask == 1)
        wshed_AR_overlap = (wshed_mask * ARmasked.ARmask).transpose('time', 'lat', 'lon')
        
        times = ARmask_full.time.values.tolist()
        landfall_wus = landfall_full['landfall_wus'].values
        
        wshed_ids = [huc4s.loc[idx, 'huc4'] for idx in range(len(huc4s))]
        watershed_data = np.zeros((len(times), len(huc4s)), dtype=int)
        
        for idx in range(len(huc4s)):
            wshed_cells = (wshed_mask == idx)
            total_count = wshed_cells.sum()
            masked_count = (wshed_AR_overlap.where(wshed_cells) >= 0).sum(dim=['lat', 'lon'])
            fraction = masked_count / total_count
            watershed_data[:, idx] = (fraction >= 0.05).astype(int).values
        
        watershed_data = watershed_data * landfall_wus[:, np.newaxis]
        
        watershed_ds = xa.Dataset(
            coords=dict(time=('time',times), watershed=('watershed', wshed_ids)),
            data_vars=dict(
                ARwshed_any=(["time"], np.array(landfall_wus)), ARwshed=(['time', 'watershed'], watershed_data)
            ),
            attrs=dict(description='A binary mask for discerning landfalling ARs over the Western US and split by watershed')
        )
        
        
        watershed_mask_ds = xa.Dataset(
            coords=dict(
                time=('time', times),
                lat=('lat', ARmask_full.lat.data),
                lon=('lon', ARmask_full.lon.data),
            ),
            data_vars=dict(
                wshed_mask=(['time','lat','lon'], wshed_AR_overlap.values)
            ),
            attrs=dict(description='The points where an AR mask overlaps the any watershed.')
        )
        
        watershed_ds.to_netcdf(f'/glade/work/tcorrie/ARdata/ARwatershed/{ardt}/ARwshed.{ardt}.{model}.{member}.nc')
        watershed_mask_ds.to_netcdf(f'/glade/derecho/scratch/tcorrie/ARdata/ARwatershedmask/{ardt}/ARwshedmsk.{ardt}.{model}.{member}.nc')
        
        del ARmask_full, landfall_full, wshed_mask, ARmasked, wshed_AR_overlap, watershed_ds, watershed_mask_ds, watershed_data
        gc.collect()

access-cm2/r5i1p1f1:
G18
TE
tARget
canesm5/r1i1p2f1:
G18
TE
tARget
cesm2/r11i1p1f1:
G18
TE
tARget
cnrm-esm2-1/r1i1p1f2:
G18
TE
tARget
ec-earth3/r1i1p1f1:
G18
TE
tARget
ec-earth3-veg/r1i1p1f1:
G18
TE
tARget
fgoals-g3/r1i1p1f1:
G18
TE
tARget
giss-e2-1-g/r1i1p1f2:
G18
TE
tARget
miroc6/r1i1p1f1:
G18
TE
tARget
mpi-esm1-2-hr/r3i1p1f1:
G18
TE
tARget
mpi-esm1-2-hr/r7i1p1f1:
G18
TE
tARget
mpi-esm1-2-lr/r7i1p1f1:
G18
TE
tARget
noresm2-mm/r1i1p1f1:
G18
TE
tARget
taiesm1/r1i1p1f1:
G18
TE
tARget
ukesm1-0-ll/r2i1p1f2:
G18
TE
tARget
cesm2-le/1011:
G18
TE
tARget
cesm2-le/1031:
G18
TE
tARget
cesm2-le/1051:
G18
TE
tARget
cesm2-le/1071:
G18
TE
tARget
cesm2-le/1091:
G18
TE
tARget
cesm2-le/1111:
G18
TE
tARget
cesm2-le/1131:
G18
TE
tARget
cesm2-le/1151:
G18
TE
tARget
cesm2-le/1171:
G18
TE
tARget
cesm2-le/1191:
G18
TE
tARget
CPU times: user 2h 22min 28s, sys: 2h 1min 46s, total: 4h 24min 15s
Wall time: 4h 27min 49s
